# 📘 부록 01. GCP VM 준비: 교안 00 전에 배포용 무료 VM 만들기

<div class="day78-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">실행 위치:</strong> 내 PC 브라우저(GCP 콘솔), 내 PC 터미널, VS Code Remote-SSH로 연 GCP VM 터미널입니다. <strong style="font-size:1em;color:#1f3a5f;">노트북 코드 셀은 없습니다.</strong><br><strong style="font-size:1em;color:#1f3a5f;">만드는 것:</strong> GCP 무료 등급의 e2-micro VM 한 대(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm</code>), 8080·8081 포트를 여는 방화벽 규칙, VS Code 원격 접속, 스왑 파일, Docker입니다.<br><strong style="font-size:1em;color:#1f3a5f;">순서:</strong> 필수입니다. 교안 00 앞에 합니다. 교안 00~03과 과제가 모두 이 VM을 씁니다.</div>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">가상 머신(VM, Virtual Machine)</strong>은 클라우드 회사의 서버 자원을 나눠 만든 컴퓨터 한 대입니다. 이 부록에서는 <strong style="font-size:1.13em;color:#1f3a5f;">GCP(Google Cloud Platform)</strong>의 VM 서비스 <strong style="font-size:1.13em;color:#1f3a5f;">Compute Engine</strong>에 VM 한 대를 만들고, 내 PC의 VS Code로 접속해 Docker까지 설치합니다. 교안 00~03은 이 VM에서 Docker 이미지를 빌드하고 서빙 API 컨테이너를 실행합니다.</p>
</div>

<div class="day78-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#fff8e1;border-left:4px solid #c77700;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#8a4b00;margin-bottom:2px;">시작 전에: 카드가 필요합니다</strong><strong style="font-size:1em;color:#8a4b00;">본인 명의 신용카드나 해외 결제가 되는 체크카드를 준비합니다.</strong> 무료로 쓰는 VM도 결제 계정(카드 등록)이 있어야 만들 수 있습니다. 선불 카드는 받지 않습니다. 다른 카드로도 안 되면 수업 전에 강사에게 알립니다(VM 없이는 교안 00부터 진행할 수 없습니다).<br><strong style="font-size:1em;color:#8a4b00;">체험 기간에는 청구되지 않습니다.</strong> 유료 계정으로 업그레이드하지 않으면 체험 중 요금은 크레딧에서 빠지고 카드로 나가지 않습니다.<br><strong style="font-size:1em;color:#8a4b00;">과정이 끝나면 VM을 삭제합니다(10절).</strong></div>

## 0. 비용 한눈에 보기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 부록의 VM과 디스크는 무료 등급(Always Free, 조건에 맞는 작은 자원을 계속 무료로 주는 프로그램) 안에 들게 만듭니다. 외부 IP도 계정당 매달 721시간까지 무료라, VM 한 대를 쓰는 이 실습은 사실상 요금이 나오지 않습니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">항목</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">요금</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">e2-micro VM 한 대(us-central1)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">무료(무료 등급)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">표준 영구 디스크 30GB</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">무료(무료 등급)</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">외부 IP(임시)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">계정당 매달 721시간까지 무료, 넘으면 시간당 <span>&#36;</span>0.005(한 대를 31일 내내 켜 두면 약 <span>&#36;</span>0.1)</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">무료 한도는 계정 전체에서 함께 씁니다.</strong> VM을 두 대 이상 켜 두면 한도를 넘으니, 실습이 없는 날은 VM을 중지해 둡니다(10절).</li>
<li style="margin:3px 0;">체험 기간에는 이 요금도 크레딧에서 빠집니다. 예상 밖 요금은 1-2의 예산 알림으로 메일을 받습니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이 부록의 흐름</strong></p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">GCP 무료 체험 가입과 예산 알림</li>
<li style="margin:3px 0;">Compute Engine API 켜기</li>
<li style="margin:3px 0;">VM 만들기</li>
<li style="margin:3px 0;">브라우저 SSH(비상용)</li>
<li style="margin:3px 0;">8080·8081 포트를 여는 방화벽 규칙</li>
<li style="margin:3px 0;">SSH 키와 VS Code Remote-SSH 접속</li>
<li style="margin:3px 0;">스왑 파일 2GB 만들기</li>
<li style="margin:3px 0;">Docker Engine 설치</li>
<li style="margin:3px 0;">실습자료 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 VM 주소 적기</li>
<li style="margin:3px 0;">VM 중지와 삭제</li>
</ol>
</div>

## 1. GCP 무료 체험 가입

### 1-1. 무료 체험 가입

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">무료 체험에 가입해 결제 계정을 만듭니다. 90일 동안 쓸 수 있는 <span>&#36;</span>300 크레딧이 함께 생기고, 이 결제 계정이 있어야 무료 등급 VM도 만들 수 있습니다.</p>
<p style="margin:6px 0;">콘솔 주소: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://console.cloud.google.com/</code></p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · 콘솔 첫 화면(시작하기)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 배너 오른쪽의 '무료로 시작하기'를 누릅니다. 체험은 계정당 한 번이라, 이미 받은 계정에는 배너가 없습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 배너 문구(<span>&#36;</span>300 크레딧 안내), '무료로 시작하기' 버튼</li></ul></div>
<img src="images/gcp_01_console_home.png" width="1050" alt="콘솔 첫 화면(시작하기). 빨간 상자: 배너 문구(&#36;300 크레딧 안내), '무료로 시작하기' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · 무료 체험판 가입 1단계(계정 정보)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 국가가 대한민국인지 확인하고 체크박스 두 개를 모두 선택한 뒤 '계속'을 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 약관 체크박스 두 개 묶음, '계속' 버튼</li></ul></div>
<img src="images/gcp_02_trial_step1_terms.png" width="1050" alt="무료 체험판 가입 1단계(계정 정보). 빨간 상자: 약관 체크박스 두 개 묶음, '계속' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 3 · 가입 2단계(결제 정보 확인)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> '결제 수단 추가'를 눌러 카드를 등록합니다. 무료 등급만 쓰더라도 결제 계정(카드)이 있어야 VM을 만들 수 있습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 결제 수단 영역, '무료로 시작하기' 버튼</li></ul></div>
<img src="images/gcp_03_trial_step2_billing.png" width="1050" alt="가입 2단계(결제 정보 확인). 빨간 상자: 결제 수단 영역, '무료로 시작하기' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 4 · '신용카드 또는 체크카드 추가' 창(빈 양식)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 본인 카드 번호, 유효기간, 보안 코드를 넣고 '카드 저장'을 누릅니다. 선불 카드는 받지 않습니다. 0~1달러 임시 승인이 보였다가 사라질 수 있습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 카드 입력 칸 세 개, '카드 저장' 버튼</li></ul></div>
<img src="images/gcp_04_trial_card_form.png" width="1050" alt="'신용카드 또는 체크카드 추가' 창(빈 양식). 빨간 상자: 카드 입력 칸 세 개, '카드 저장' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 5 · 카드 저장 뒤 2단계 화면</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 카드가 들어갔는지 확인하고 '무료로 시작하기'를 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 결제 수단 영역, '무료로 시작하기' 버튼</li></ul></div>
<img src="images/gcp_05_trial_card_registered.png" width="1050" alt="카드 저장 뒤 2단계 화면. 빨간 상자: 결제 수단 영역, '무료로 시작하기' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 6 · 체험판 시작 직후 환영 화면</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 혜택(90일 동안 쓰는 <span>&#36;</span>300 크레딧)을 확인하고 'VM 만들기'를 누릅니다. 콘솔 위쪽의 '업그레이드'는 유료 계정으로 바꾸는 버튼이라 누르지 않습니다. 업그레이드하지 않으면 체험 중 카드로 청구되지 않습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 혜택 세 줄(제품 액세스, 90일 크레딧, 청구 없음), 'VM 만들기' 버튼</li></ul></div>
<img src="images/gcp_06_trial_welcome.png" width="1050" alt="체험판 시작 직후 환영 화면. 빨간 상자: 혜택 세 줄(제품 액세스, 90일 크레딧, 청구 없음), 'VM 만들기' 버튼">

### 1-2. 예산 알림 만들기(권장)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">예상 밖 요금이 나오면 메일로 알 수 있게 예산 알림을 하나 만듭니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GCP 콘솔 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 왼쪽 메뉴 Cost control &gt; Budgets &amp; caps &gt; Create new(위쪽 경로에는 '결제 / 예산 및 알림 / 예산 만들기'로 보입니다)</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://console.cloud.google.com/billing/budgets</code></li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · 예산 만들기 3단계(금액)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 1단계 '정의'에서 '알림만'을 고르고 이름을 넣습니다. 2단계 '범위'는 기본값 그대로 '다음'을 누릅니다. 3단계 '금액'의 목표 금액에 결제 계정 통화로 작은 금액(약 7,000원, 약 5달러)을 넣고 '다음'을 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 목표 금액 칸</li></ul></div>
<img src="images/gcp_41_budget_create_form.png" width="1050" alt="결제 > Budgets & caps > Create new 의 예산 만들기 3단계 '금액'. 빨간 상자: 목표 금액 칸">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · 예산 만들기 4단계(작업)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 4단계 '작업'의 알림 기준(50%, 90%, 100%)과 결제 관리자 이메일 알림은 기본값 그대로 두고 '완료'를 누릅니다. 예산은 사용을 막지 않고, 기준에 닿으면 메일로 알리기만 합니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 알림 기준 세 줄, 이메일 알림 체크박스, '완료' 버튼</li></ul></div>
<img src="images/gcp_42_budget_thresholds.png" width="1050" alt="4단계 '작업'. 빨간 상자: 알림 기준 세 줄, 이메일 알림 체크박스, '완료' 버튼">

## 2. Compute Engine API 켜기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VM 서비스(Compute Engine)를 이 프로젝트에서 쓸 수 있게 켭니다. 처음 VM을 만들려고 하면 이 화면이 먼저 나옵니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · Compute Engine API 제품 세부정보</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> '사용'을 누르고 1~2분 기다립니다. GCP 서비스는 프로젝트(자원과 결제를 묶는 단위)마다 그 서비스의 API를 켜야 쓸 수 있습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> '사용' 버튼</li></ul></div>
<img src="images/gcp_07_compute_api_enable.png" width="1050" alt="Compute Engine API 제품 세부정보. 빨간 상자: '사용' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · API 및 서비스 &gt; Compute Engine API</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 상태가 '사용 설정됨'인지 확인합니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 상태 '사용 설정됨'</li></ul></div>
<img src="images/gcp_08_compute_api_enabled.png" width="1050" alt="API 및 서비스의 Compute Engine API 세부정보. 빨간 상자: 상태 '사용 설정됨'">

## 3. VM 만들기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">무료 등급 조건에 맞춰 VM 한 대를 만듭니다. 값마다 바꾸는 이유는 3-2의 화면마다 적었습니다.</p>
</div>

### 3-1. 설정 값

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">값</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이름</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">리전</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">us-central1</code>(아이오와), 영역은 기본값</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">머신 유형</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">시리즈 E2, <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">e2-micro</code></strong></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">부팅 디스크</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이미지는 기본값 Debian GNU/Linux 13, 유형과 크기만 <strong style="font-size:1.13em;color:#1f3a5f;">표준 영구 디스크 30GB</strong></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">네트워크 태그</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-api</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">HTTP, HTTPS 트래픽 허용</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">선택하지 않음</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">외부 IP</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">기본값(임시)</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">표에 없는 칸은 기본값 그대로 둡니다. 교안과 과제가 이름 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm</code>과 태그 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-api</code>를 그대로 쓰므로 바꾸지 않습니다.</p>
</div>

### 3-2. 콘솔에서 만들기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">콘솔 주소: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://console.cloud.google.com/compute/instances</code></p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · 빈 VM 인스턴스 목록</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 왼쪽 'VM 인스턴스'를 열고 '인스턴스 만들기'를 누릅니다. Compute Engine은 VM 한 대를 인스턴스라고 부릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 왼쪽 메뉴 'VM 인스턴스', '인스턴스 만들기' 버튼</li></ul></div>
<img src="images/gcp_09_vm_instances_empty.png" width="1050" alt="Compute Engine > VM 인스턴스 목록(비어 있음). 빨간 상자: 왼쪽 메뉴 'VM 인스턴스', '인스턴스 만들기' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · 인스턴스 만들기 &gt; 머신 구성 위쪽</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 이름에 serving-vm을 넣고 리전(데이터센터 지역)이 us-central1(아이오와)인지 확인합니다. 무료 등급은 us-central1을 포함한 미국 세 리전만 해당합니다. 영역은 그대로 둡니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 이름 칸, 리전 칸</li></ul></div>
<img src="images/gcp_10_vm_name_region.png" width="1050" alt="인스턴스 만들기 > 머신 구성 위쪽. 빨간 상자: 이름 칸, 리전 칸">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 3 · E2 시리즈의 머신 유형 목록</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 시리즈 E2를 고르고 머신 유형 목록에서 e2-micro(메모리 1GB)를 고릅니다. 기본값 e2-medium은 무료가 아닙니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> E2 행, e2-micro 항목</li></ul></div>
<img src="images/gcp_11_vm_machine_type_menu.png" width="1050" alt="머신 시리즈 표에서 E2가 선택된 상태로 머신 유형 목록을 연 화면. 빨간 상자: E2 행, e2-micro 항목">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 4 · e2-micro를 고른 뒤 월별 예상 가격</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 머신 유형이 e2-micro인지 확인합니다. 오른쪽 월별 예상 가격은 무료 등급을 빼지 않은 정가라 이 금액이 청구되지는 않습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 머신 유형 칸, 월별 예상 가격 패널</li></ul></div>
<img src="images/gcp_12_vm_e2_micro_cost.png" width="1050" alt="e2-micro를 고른 뒤 머신 유형 칸과 오른쪽 월별 예상 가격(디스크는 아직 기본값). 빨간 상자: 머신 유형 칸, 월별 예상 가격 패널">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 5 · 왼쪽 'OS 및 스토리지' 화면</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 왼쪽 'OS 및 스토리지'를 누릅니다. 운영체제 이미지(Debian 13)는 그대로 두고, 디스크만 바꾸려고 '변경'을 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> '변경' 버튼</li></ul></div>
<img src="images/gcp_13_vm_os_storage.png" width="1050" alt="왼쪽 'OS 및 스토리지' 화면. 빨간 상자: '변경' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 6 · 부팅 디스크 창</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 부팅 디스크 유형에서 '표준 영구 디스크'를 고릅니다. 기본값 '균형 있는 영구 디스크'는 무료가 아니고, 무료 등급은 표준 영구 디스크 30GB까지입니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> '표준 영구 디스크' 항목</li></ul></div>
<img src="images/gcp_14_vm_disk_type_menu.png" width="1050" alt="부팅 디스크 창. 빨간 상자: '표준 영구 디스크' 항목">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 7 · 다 채운 부팅 디스크 창</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 크기에 30을 넣고 '선택'을 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 디스크 유형·크기 묶음, '선택' 버튼</li></ul></div>
<img src="images/gcp_15_vm_boot_disk_set.png" width="1050" alt="부팅 디스크 창을 다 채운 모습(Debian 13 그대로, 표준 영구 디스크, 30GB). 빨간 상자: 디스크 유형·크기 묶음, '선택' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 8 · 바뀐 OS 및 스토리지 요약과 월별 예상 가격</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 유형 '새로운 표준 영구 디스크', 크기 30GB, 이미지 Debian 13을 확인합니다. 디스크 가격이 0이 된 것도 봅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 디스크 요약 줄, 월별 예상 가격 패널</li></ul></div>
<img src="images/gcp_16_vm_os_storage_done.png" width="1050" alt="바뀐 OS 및 스토리지 요약과 월별 예상 가격. 빨간 상자: 디스크 요약 줄, 월별 예상 가격 패널">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 9 · 왼쪽 '네트워킹' 화면</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 왼쪽 '네트워킹'을 누르고 네트워크 태그에 serving-api를 넣은 뒤 Enter를 누릅니다. 태그는 VM에 붙이는 이름표로, 5절 방화벽 규칙이 이 태그가 붙은 VM에만 8080, 8081 포트를 엽니다. HTTP, HTTPS 트래픽 허용(80, 443 포트)은 체크하지 않습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 방화벽 체크박스 세 개(해제 상태), 네트워크 태그 칸</li></ul></div>
<img src="images/gcp_17_vm_networking_tag.png" width="1050" alt="왼쪽 '네트워킹' 화면. 빨간 상자: 방화벽 체크박스 세 개(해제 상태), 네트워크 태그 칸">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 10 · 펼친 네트워크 인터페이스(nic0)</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 아래로 내려 네트워크 인터페이스의 default 줄을 펼치고, 외부 IPv4 주소가 '임시'인지 확인한 뒤 '완료'를 누릅니다. 임시 주소는 VM이 켜져 있는 동안만 사용 시간이 계산되고, 중지하면 반납됩니다(다시 켜면 주소가 바뀔 수 있습니다). 외부 IP는 계정당 매달 721시간까지 무료라 VM 한 대를 한 달 내내 켜 두어도 IP 요금은 거의 나오지 않습니다. 고정 IP는 VM에서 떼어 두면 시간당 <span>&#36;</span>0.01이 나오므로 예약하지 않습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 외부 IPv4 주소 '임시' 칸, 월별 예상 가격 패널(외부 IP 줄이 없음)</li></ul></div>
<img src="images/gcp_18_vm_external_ip.png" width="1050" alt="네트워킹 > 네트워크 인터페이스(nic0)를 펼친 화면. 빨간 상자: 외부 IPv4 주소 '임시' 칸, 월별 예상 가격 패널(외부 IP 줄이 없음)">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 11 · 설정을 마친 머신 구성 화면과 최종 월별 예상 가격</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 왼쪽 요약(e2-micro, us-central1, Debian 13)을 확인하고 '만들기'를 누릅니다. 예상 가격은 무료 등급을 빼지 않은 금액이고, 외부 IP는 이 패널에 줄이 없습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 월별 예상 가격 패널, '만들기' 버튼</li></ul></div>
<img src="images/gcp_19_vm_create_button.png" width="1050" alt="설정을 마친 머신 구성 화면과 최종 월별 예상 가격. 빨간 상자: 월별 예상 가격 패널, '만들기' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 12 · serving-vm이 실행 중인 VM 인스턴스 목록</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 외부 IP를 적어 둡니다. 6-3의 SSH 설정과 9절의 .env에 씁니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 외부 IP, 'SSH' 버튼</li></ul></div>
<img src="images/gcp_20_vm_list_running.png" width="1050" alt="VM 인스턴스 목록에 serving-vm이 실행 중(초록 체크)으로 나온 화면. 빨간 상자: 외부 IP, 'SSH' 버튼">

## 4. 브라우저 SSH(비상용)

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VS Code 접속(6절)이 막혔을 때는 VM 목록 줄의 <strong style="font-size:1.13em;color:#1f3a5f;">SSH</strong> 버튼으로 브라우저에서 VM 터미널을 열 수 있습니다. 브라우저 SSH는 Google 계정 이름의 다른 사용자로 접속하므로, 실습 명령은 6절에서 접속한 VS Code 터미널에서 실행합니다.</p>
</div>

## 5. 방화벽 규칙

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">인터넷에서 서빙 API(8080)와 쇼핑 상담 API(8081)에 들어올 수 있게 방화벽 규칙을 하나 만듭니다. <strong style="font-size:1.13em;color:#1f3a5f;">방화벽 규칙(firewall rule)</strong>은 어떤 주소에서 어떤 포트로 들어오는 연결을 VM에 들일지 정하는 규칙입니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GCP 콘솔 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://console.cloud.google.com/net-security/firewall-manager/firewall-policies/list</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">Compute Engine 화면 위쪽 검색창에 <strong style="font-size:1.13em;color:#1f3a5f;">방화벽</strong>을 입력하고 결과 맨 위의 <strong style="font-size:1.13em;color:#1f3a5f;">방화벽</strong>(VPC 네트워크)을 누릅니다(화면 1).</li>
<li style="margin:3px 0;">VPC 방화벽 규칙 목록에서 <strong style="font-size:1.13em;color:#1f3a5f;">방화벽 규칙 만들기</strong>를 누릅니다(화면 2).</li>
<li style="margin:3px 0;">양식을 채우고 <strong style="font-size:1.13em;color:#1f3a5f;">만들기</strong>를 누릅니다(화면 3).</li>
<li style="margin:3px 0;">목록에서 새 규칙을 확인합니다(화면 4).</li>
</ol>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · 검색창의 방화벽 검색 결과</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> Compute Engine 화면 위쪽 검색창을 누르고 방화벽을 입력합니다. 뒤에 '규칙' 같은 글자가 붙어 보이는 것은 자동 완성이라 그대로 둡니다. 결과 맨 위의 '방화벽'(제품 페이지, VPC 네트워크)을 누릅니다. 왼쪽 메뉴 제목이 '네트워크 보안'으로 바뀐 목록이 열리면 맞게 온 것입니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 검색창, 결과 맨 위 '방화벽'(VPC 네트워크)</li></ul></div>
<img src="images/gcp_26a_search_firewall.png" width="1050" alt="Compute Engine 화면 위쪽 검색창의 방화벽 검색 결과. 빨간 상자: 검색창, 결과 맨 위 '방화벽'(VPC 네트워크)">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · VPC 방화벽 규칙 목록</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 목록에 default-allow-ssh(tcp:22 허용)가 있는지 확인합니다. 이 기본 규칙 덕분에 6절의 SSH 접속이 됩니다. 위쪽 '방화벽 규칙 만들기'를 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> default-allow-ssh 행, '방화벽 규칙 만들기' 버튼</li></ul></div>
<img src="images/gcp_26_firewall_default_allow_ssh.png" width="1050" alt="네트워크 보안 > 방화벽 정책의 VPC 방화벽 규칙 목록. 빨간 상자: default-allow-ssh 행, '방화벽 규칙 만들기' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 3 · 방화벽 규칙 만들기 양식</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 이름 allow-serving-api, 트래픽 방향 '수신', 작업 '허용'으로 둡니다. 대상은 '지정된 대상 태그'를 고르고 serving-api를 넣습니다(3-2에서 VM에 붙인 태그와 같아야 그 VM에 적용됩니다). 소스 IPv4 범위는 0.0.0.0/0(모든 주소, 내 PC와 GitHub Actions가 어디서든 닿도록), TCP 포트는 8080,8081을 넣고 '만들기'를 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 이름 칸, 대상 태그 칸, 소스 IPv4 범위 칸, TCP 포트 칸</li></ul></div>
<img src="images/gcp_26b_firewall_create_form.png" width="1050" alt="방화벽 규칙 만들기 양식. 빨간 상자: 이름 칸, 대상 태그 칸, 소스 IPv4 범위 칸, TCP 포트 칸">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 4 · allow-serving-api가 생긴 규칙 목록</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 목록에 allow-serving-api 행(대상 serving-api, tcp:8080,8081, 허용)이 생겼는지 확인합니다. 대상 칸이 '전체 적용'이면 대상 태그가 빠진 것입니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> allow-serving-api 행</li></ul></div>
<img src="images/gcp_26c_firewall_rule_created.png" width="1050" alt="규칙을 만든 뒤의 VPC 방화벽 규칙 목록. 빨간 상자: allow-serving-api 행">

## 6. SSH 키와 VS Code Remote-SSH 접속

### 6-1. SSH 키 만들기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">내 PC에서 이 VM 전용 SSH 키 쌍(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/.ssh/gcp_vm</code>)을 만듭니다. 공개 키는 6-2에서 VM에 등록하고, 개인 키는 내 PC에만 둡니다.</p>
<p style="margin:6px 0;">VM에서 쓸 리눅스 사용자 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">appuser</code>로 하겠습니다. 다른 이름을 쓸 때는 영문 소문자로 짓습니다. 단, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">root</code>는 사용할 수 없습니다. 이 부록과 교안 03에서 같은 이름을 씁니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
mkdir $HOME/.ssh

ssh-keygen -t ed25519 -f $HOME/.ssh/gcp_vm -C appuser

cat $HOME/.ssh/gcp_vm.pub
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">mkdir &#36;HOME/.ssh</code>: 키를 둘 폴더를 만듭니다. 이미 있다는 오류가 나오면 폴더가 있는 것이니 그대로 둡니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ssh-keygen -t ed25519 -f &#36;HOME/.ssh/gcp_vm -C appuser</code>: VM 전용 키 쌍을 만듭니다. 암호를 물으면 Enter를 두 번 누릅니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-C</code>의 이름은 공개 키 한 줄의 끝 칸이 되고, GCP는 이 칸을 VM 사용자 이름으로 읽습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-t ed25519</code>는 키 종류(Ed25519)를, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-f</code>는 개인 키를 저장할 파일 경로를 정합니다. 공개 키는 같은 경로에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.pub</code>를 붙인 파일로 생깁니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cat &#36;HOME/.ssh/gcp_vm.pub</code>: 공개 키 한 줄을 출력합니다.</li>
<li style="margin:3px 0;">다른 이름을 쓰려면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">appuser</code>를 그 이름으로 바꾸고, 이 부록과 교안 03의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">appuser</code>도 모두 같은 이름으로 씁니다.</li>
<li style="margin:3px 0;">복사할 것은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cat</code>이 출력한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ssh-ed25519</code>로 시작하는 한 줄 전체입니다. 키를 만들 때 나온 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SHA256:</code>으로 시작하는 줄은 키를 구분하는 요약값(지문)이라, GCP에 넣으면 '키가 잘못되었습니다'가 뜹니다.</li>
<li style="margin:3px 0;">macOS 터미널과 Windows PowerShell에서 같은 명령입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;HOME</code>은 두 곳 모두 홈 폴더를 가리킵니다.</li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · 내 PC 터미널의 SSH 키 생성과 공개키 출력</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 암호(passphrase)를 두 번 물으면 두 번 모두 Enter를 누르고, cat이 출력한 ssh-ed25519 한 줄 전체를 복사합니다. 개인 키 파일 gcp_vm(.pub가 없는 파일)은 내 PC에만 둡니다. 화면은 -N &quot;&quot;로 암호 질문 없이 실행한 모습입니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> ssh-keygen 명령 줄, cat 명령과 공개키 줄</li></ul></div>
<img src="images/gcp_22_vscode_ssh_keygen.png" width="1050" alt="VS Code 내 PC 터미널에서 수업용 SSH 키를 만들고 공개키를 출력한 화면. 빨간 상자: ssh-keygen 명령 줄, cat 명령과 공개키 줄">

### 6-2. VM에 공개 키 등록

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">복사한 공개 키를 VM 설정에 등록합니다. 등록하면 VM이 내 PC의 개인 키를 확인하고 암호 없이 접속을 허락합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GCP 콘솔 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> Compute Engine &gt; VM 인스턴스 &gt; serving-vm &gt; 수정 &gt; SSH 키 &gt; 항목 추가 &gt; 저장</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://console.cloud.google.com/compute/instances</code></li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · VM 인스턴스 목록</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> VM 목록에서 이름 serving-vm을 눌러 세부정보로 들어갑니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> serving-vm 이름 링크</li></ul></div>
<img src="images/gcp_23_vm_open_detail.png" width="1050" alt="VM 인스턴스 목록. 빨간 상자: serving-vm 이름 링크">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · serving-vm 세부정보 화면</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 위쪽 '수정'을 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> '수정' 버튼</li></ul></div>
<img src="images/gcp_24_vm_detail_edit.png" width="1050" alt="serving-vm 세부정보 화면. 빨간 상자: '수정' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 3 · 인스턴스 수정 &gt; 보안 및 액세스 &gt; SSH 키</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> SSH 키에서 '항목 추가'를 누르고 복사한 공개 키 한 줄을 붙여 넣은 뒤 맨 아래 '저장'을 누릅니다. 브라우저 SSH를 쓴 적이 있으면 Google 계정 이름으로 끝나는 키 줄이 이미 있습니다. 그 줄은 그대로 두고 새 칸에 붙여 넣습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 새 SSH 키 칸, '저장' 버튼</li></ul></div>
<img src="images/gcp_25_vm_add_ssh_key.png" width="1050" alt="인스턴스 수정 > 보안 및 액세스 > SSH 키. 빨간 상자: 새 SSH 키 칸, '저장' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 4 · 저장 뒤 세부정보의 SSH 키 표</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 사용자 이름이 appuser인 줄이 생겼는지 확인합니다. GCP는 공개 키 끝의 이름으로 VM에 그 사용자를 만듭니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> appuser 키 행</li></ul></div>
<img src="images/gcp_27_vm_ssh_key_saved.png" width="1050" alt="저장 뒤 세부정보의 SSH 키 표. 빨간 상자: appuser 키 행">

### 6-3. SSH 설정 파일에 VM 등록

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VS Code <strong style="font-size:1.13em;color:#1f3a5f;">Remote - SSH</strong> 확장(게시자 Microsoft)이 VM을 별명 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm</code>으로 찾도록 SSH 설정 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/.ssh/config</code>)에 등록합니다. 확장이 없으면 Extensions에서 설치합니다. 교안의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">scp</code> 명령도 이 별명을 씁니다.</p>
<p style="margin:6px 0;">명령 팔레트(macOS <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Cmd+Shift+P</code>, Windows <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+Shift+P</code>)에서 <strong style="font-size:1.13em;color:#1f3a5f;">Remote-SSH: Open SSH Configuration File...</strong>을 고르고, 아래 네 줄을 파일 맨 아래에 붙여 넣습니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · ~/.ssh/config 에 더할 내용</strong></div>

```text
Host serving-vm
    HostName <VM 외부 IP>
    User appuser
    IdentityFile ~/.ssh/gcp_vm
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;VM 외부 IP&gt;</code>는 3-2에서 적어 둔 외부 IP로 바꿉니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IdentityFile</code>은 Windows에서도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/.ssh/gcp_vm</code> 그대로 씁니다.</li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · SSH 설정 파일 선택 목록</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 첫 줄(~/.ssh/config)을 고릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 첫 번째 항목</li></ul></div>
<img src="images/gcp_28_vscode_open_ssh_config.png" width="1050" alt="VS Code 명령 팔레트 'Remote-SSH: Open SSH Configuration File...' 다음의 파일 선택 목록. 빨간 상자: 첫 번째 항목">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · Host 항목을 적은 ~/.ssh/config</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 위의 네 줄을 파일 맨 아래에 붙여 넣고 저장합니다. 이제 별명 serving-vm 하나로 주소, 사용자, 키 파일을 함께 가리킵니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> Host 항목 네 줄</li></ul></div>
<img src="images/gcp_29_vscode_ssh_config_host.png" width="1050" alt="~/.ssh/config 에 Host 항목을 적은 화면. 빨간 상자: Host 항목 네 줄">

### 6-4. 접속

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VS Code를 VM에 연결합니다. 연결하면 창이 두 개가 됩니다. 실습자료 폴더 창은 노트북과 내 PC 터미널에, 새로 열린 VM 창은 VM의 파일과 VM 터미널에 씁니다. 교안 00~03은 두 창을 함께 씁니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · 명령 팔레트의 Connect to Host 입력</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 명령 팔레트에서 'Remote-SSH: Connect to Host...'를 고릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 'Remote-SSH: Connect to Host...' 항목</li></ul></div>
<img src="images/gcp_30_vscode_connect_to_host_cmd.png" width="1050" alt="명령 팔레트에 Remote-SSH: Connect to Host 를 입력한 화면. 빨간 상자: 'Remote-SSH: Connect to Host...' 항목">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · 호스트 목록</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> serving-vm을 고릅니다. VM에 연결된 새 VS Code 창이 열립니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> serving-vm 항목</li></ul></div>
<img src="images/gcp_31_vscode_host_list.png" width="1050" alt="config 에 적은 호스트 목록. 빨간 상자: serving-vm 항목">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 3 · 첫 접속 때 나오는 호스트 지문 확인 입력창</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 처음 접속할 때만 나옵니다. 이 서버의 식별 값(지문)을 기억할지 묻는 것이라 yes를 입력하고 Enter를 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 지문 확인 입력창</li></ul></div>
<img src="images/gcp_32_vscode_host_fingerprint.png" width="1050" alt="첫 접속 때 나오는 호스트 지문 확인 입력창. 빨간 상자: 지문 확인 입력창">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 4 · 연결된 VM 창</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 왼쪽 아래에 SSH: serving-vm이 보이면 접속된 것입니다. Terminal &gt; New Terminal로 VM 터미널을 엽니다(교안에서 'GCP VM 터미널'이라고 부르는 곳).</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 'SSH: serving-vm' 표시, 'Connected to remote.' 문구</li></ul></div>
<img src="images/gcp_33_vscode_connected.png" width="1050" alt="연결된 VS Code 창. 빨간 상자: 'SSH: serving-vm' 표시, 'Connected to remote.' 문구">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# 접속한 사용자 이름, 커널, 메모리를 확인합니다
whoami
uname -a
free -h
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 5 · 원격 터미널의 확인 명령 결과</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 위 명령의 결과가 이 화면과 같은 모양인지 봅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> uname 결과 줄, free -h의 Mem 줄</li></ul></div>
<img src="images/gcp_34_vscode_remote_terminal.png" width="1050" alt="원격 터미널에서 uname -a, free -h, df -h / 실행 결과. 빨간 상자: uname 결과 줄, free -h의 Mem 줄">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">whoami</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 접속한 사용자 이름을 출력합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uname -a</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">커널 이름, 버전, 호스트 이름 같은 시스템 정보를 모두(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-a</code>, all) 출력합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">free -h</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">메모리와 스왑의 전체, 사용, 남은 양을 사람이 읽기 쉬운 단위(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-h</code>, Mi, Gi)로 보여 줍니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">whoami</code>가 6-1의 사용자 이름, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">uname -a</code>에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">deb13</code>:</strong> 등록한 키로 그 사용자에, 3-2에서 고른 Debian 13 VM에 접속했습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">free -h</code>의 available이 작습니다:</strong> VM 안의 VS Code 서버도 메모리를 씁니다. 이대로는 이미지 빌드에 모자라므로 7절에서 스왑을 만듭니다.</li>
</ul>
</div>

## 7. 스왑 파일 2GB 만들기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">메모리가 1GB라 이미지를 빌드하다 멈출 수 있어, 디스크 2GB를 메모리처럼 쓰는 <strong style="font-size:1.13em;color:#1f3a5f;">스왑(swap)</strong> 파일을 만듭니다. 메모리보다 느리지만 잠깐 메모리를 많이 쓰는 빌드가 멈추지 않게 합니다. Docker를 설치하기 전에 VM 터미널에서 한 번만 실행합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# 2GB 파일을 만들고 root 만 읽고 쓰게 합니다
sudo fallocate -l 2G /swapfile
sudo chmod 600 /swapfile

# 스왑 공간으로 만들고 바로 켭니다
sudo mkswap /swapfile
sudo swapon /swapfile

# VM 을 다시 켜도 스왑이 켜지도록 /etc/fstab 에 한 줄을 더합니다(두 번 실행하면 같은 줄이 두 번 들어갑니다)
echo '/swapfile none swap sw 0 0' | sudo tee -a /etc/fstab

# 메모리와 스왑 크기를 확인합니다
free -h
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sudo 명령</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">뒤의 명령을 관리자(root) 권한으로 실행합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">fallocate -l 2G 파일</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">크기 2GB(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-l</code>, length)의 파일을 디스크에 미리 잡아 만듭니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">chmod 600 파일</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">파일 주인(root)만 읽고 쓸 수 있게 권한을 바꿉니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">mkswap 파일</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">파일을 스왑 공간으로 쓸 수 있게 준비합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">swapon 파일</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">준비한 스왑 공간을 바로 켭니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">echo '내용'</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">따옴표 안의 내용을 출력합니다. 여기서는 출력을 파이프로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>에 넘깁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sudo tee -a 파일</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">넘겨받은 내용을 root 권한으로 파일 끝에 덧붙입니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-a</code>, append). <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-a</code>가 없으면 파일 내용을 덮어씁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/swapfile none swap sw 0 0</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/etc/fstab</code>(부팅 때 연결할 디스크 목록)에 적는 줄로, 부팅할 때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/swapfile</code>을 스왑으로 켜라는 뜻입니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Swap:</code> 줄의 total이 약 2.0Gi:</strong> 스왑이 켜졌습니다.</li>
</ul>
</div>

## 8. Docker Engine 설치

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">Docker 공식 문서의 Debian 설치 방법대로 Docker Engine과 compose를 설치합니다. 교안 00에서 저장소를 받을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">git</code>도 함께 설치합니다.</p>
</div>

### 8-1. Docker apt 저장소 등록

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">Debian의 패키지 설치 도구 <strong style="font-size:1.13em;color:#1f3a5f;">apt</strong>가 Docker 패키지를 받아 올 주소(apt 저장소)를 등록합니다. 아래 블록 전체를 한 번에 붙여 넣어 실행합니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">EOF</code>까지가 파일 하나의 내용입니다).</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# 패키지 목록을 갱신하고 인증서, curl, git 을 설치합니다
sudo apt update
sudo apt install -y ca-certificates curl git

# 받은 패키지가 Docker 가 만든 것인지 확인할 때 쓰는 GPG 키를 받습니다
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/debian/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

# Docker apt 저장소를 apt 의 저장소 목록에 더합니다(Suites 에는 /etc/os-release 의 버전 이름 trixie 가 들어갑니다)
sudo tee /etc/apt/sources.list.d/docker.sources <<EOF
Types: deb
URIs: https://download.docker.com/linux/debian
Suites: $(. /etc/os-release && echo "$VERSION_CODENAME")
Components: stable
Architectures: $(dpkg --print-architecture)
Signed-By: /etc/apt/keyrings/docker.asc
EOF

# 새 저장소의 패키지 목록을 받습니다
sudo apt update
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">apt update</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">등록된 apt 저장소에서 최신 패키지 목록을 받아 옵니다. 패키지를 설치하지는 않습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">apt install -y 패키지...</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">패키지를 설치합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-y</code>는 설치할지 묻는 질문에 자동으로 yes라고 답합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">install -m 0755 -d 폴더</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">폴더를 만들고(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-d</code>), 권한을 0755(주인은 읽기, 쓰기, 들어가기, 나머지는 읽기와 들어가기만)로 정합니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-m</code>).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">curl -fsSL 주소 -o 파일</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">주소의 파일을 받아 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-o</code> 뒤 경로에 저장합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-f</code>는 서버 오류면 실패로 끝내고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-s</code>는 진행 표시를 숨기고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-S</code>는 숨겨도 오류는 보여 주고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-L</code>은 다른 주소로 넘기면 따라갑니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">chmod a+r 파일</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">모든 사용자(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">a</code>)에게 읽기(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">r</code>) 권한을 더합니다. apt가 키 파일을 읽을 수 있게 합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sudo tee 파일 &lt;&lt;EOF</code> ... <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">EOF</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">두 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">EOF</code> 사이의 줄을 root 권한으로 파일에 씁니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-a</code>가 없어 파일을 새로 씁니다).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;(명령)</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">괄호 안 명령의 출력으로 그 자리를 채웁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">. /etc/os-release &amp;&amp; echo &quot;&#36;VERSION_CODENAME&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">OS 정보 파일을 읽어 들인 뒤 Debian 버전 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">trixie</code>)을 출력합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dpkg --print-architecture</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이 VM의 CPU 아키텍처 이름(e2-micro는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">amd64</code>)을 출력합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Signed-By</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이 저장소의 패키지를 위에서 받은 키(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker.asc</code>)로 확인하게 합니다.</td>
</tr>
</tbody>
</table>
</div>

### 8-2. Docker 패키지 설치

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">설치가 끝나면 Docker가 바로 켜지고, VM을 다시 켜도 자동으로 켜집니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# Docker Engine, 명령 도구, 빌드 도구(buildx), compose(교안 02 의 docker compose)를 설치합니다
sudo apt install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">패키지</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker-ce</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">Docker Engine입니다. 이미지와 컨테이너를 관리하는 서버 프로그램입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker-ce-cli</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">터미널에서 쓰는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker</code> 명령 도구입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">containerd.io</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">Docker Engine 아래에서 컨테이너를 실제로 실행하고 관리하는 런타임입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker-buildx-plugin</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker build</code>가 쓰는 빌드 도구(buildx)입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker-compose-plugin</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">여러 컨테이너를 파일 하나로 함께 띄우는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose</code> 명령입니다.</td>
</tr>
</tbody>
</table>
</div>

### 8-3. sudo 없이 docker 쓰기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">내 사용자를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker</code> 그룹에 넣어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sudo</code> 없이 docker를 쓰게 합니다. 교안 03의 자동 배포도 이 사용자 권한으로 docker를 실행합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# 지금 사용자를 docker 그룹에 더합니다
sudo usermod -aG docker $USER
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usermod -aG 그룹 사용자</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">사용자를 그룹에 더합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-G</code>는 넣을 그룹이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-a</code>(append)는 이미 속한 그룹을 그대로 두고 더합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-a</code> 없이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-G</code>만 쓰면 다른 보조 그룹에서 빠집니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;USER</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 사용자 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">appuser</code>)이 들어가는 환경 변수입니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">바로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sudo</code> 없이 docker가 되는지 확인합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# 그룹 목록에 docker 가 있는지 확인합니다
groups

# sudo 없이 시험용 이미지를 받아 컨테이너로 실행합니다(--rm 은 끝난 컨테이너를 바로 지웁니다)
docker run --rm hello-world
docker compose version
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">명령, 옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">groups</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 사용자가 속한 그룹 목록을 출력합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker run --rm hello-world</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">시험용 이미지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">hello-world</code>를 받아 컨테이너로 실행합니다. 컨테이너는 안내 문장을 출력하고 끝나며, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--rm</code>이 끝난 컨테이너를 바로 지웁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">docker compose version</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">설치된 compose 플러그인의 버전을 출력합니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">groups</code> 줄에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">docker</code>, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Hello from Docker!</code>와 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Docker Compose version</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sudo</code> 없이 docker와 compose를 쓸 수 있습니다. 9절로 넘어갑니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">permission denied</code>로 끝남:</strong> 그룹 변경은 새로 맺은 연결부터 적용되는데, 지금 터미널은 그룹을 바꾸기 전에 맺은 연결이라 그렇습니다. 아래처럼 다시 접속한 뒤 위 명령을 다시 실행합니다.</li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">명령 팔레트에서 <strong style="font-size:1.13em;color:#1f3a5f;">Remote-SSH: Kill VS Code Server on Host...</strong>를 고르고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm</code>을 고릅니다. 원격 연결이 끊깁니다.</li>
<li style="margin:3px 0;">명령 팔레트에서 <strong style="font-size:1.13em;color:#1f3a5f;">Remote-SSH: Connect to Host...</strong>로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm</code>에 다시 접속하고, 새 VM 창에서 새 터미널을 엽니다(끊긴 창은 닫습니다).</li>
<li style="margin:3px 0;">그래도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">permission denied</code>로 끝나면 VM 터미널에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sudo reboot</code>로 VM을 다시 시작하고 1분 뒤 다시 접속합니다.</li>
</ol>
</div>

## 9. 실습자료 .env에 VM 주소 적기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안과 과제의 노트북은 내 PC에서 VM의 API(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://&lt;VM 외부 IP&gt;:8080</code>)로 요청하고, 이 주소를 실습자료 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_HOST</code>에서 읽습니다. 실습자료 폴더 창(6-4)에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 열고 맨 아래에 한 줄을 더해 저장합니다(파일이 없으면 새로 만듭니다).</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · 실습자료/.env 에 더할 줄</strong></div>

```text
VM_HOST=<VM 외부 IP>
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://</code>와 포트 번호 없이 숫자 주소만 적습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>는 키와 주소를 모아 두는 파일이라 GitHub에 올리지 않습니다.</li>
</ul>
</div>

## 10. VM 중지와 삭제

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VM 한 대는 켜 두어도 무료 한도 안이지만, 실습이 없는 날은 VM을 중지하고 과정이 끝나면 삭제합니다.</p>
</div>

### 10-1. 중지와 다시 시작

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GCP 콘솔 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> Compute Engine &gt; VM 인스턴스 &gt; serving-vm 줄의 점 세 개 &gt; 중지(다시 켤 때는 시작/재개)</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://console.cloud.google.com/compute/instances</code></li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · serving-vm 줄의 점 세 개 메뉴</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 그날 실습을 마치면 serving-vm 줄의 점 세 개를 누르고 '중지'를 고릅니다. 바로 아래 '정지'(Suspend)는 메모리 내용을 보관하는 다른 기능이라 고르지 않습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> '중지' 메뉴</li></ul></div>
<img src="images/gcp_38_vm_stop_menu.png" width="1050" alt="실행 중인 serving-vm 행의 점 세 개(옵션 더보기) 메뉴. 빨간 상자: '중지' 메뉴">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · 중지 확인 창</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> '단계적 종료 건너뛰기'는 체크하지 않은 채 '중지'를 누릅니다. 중지하면 임시 외부 IP가 반납되어 사용 시간 계산이 멈추고, 디스크(스왑, Docker, 받은 저장소)는 그대로 남습니다. 표준 디스크 30GB는 무료 등급 안이라 중지해 둔 동안 나오는 요금이 없습니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> '중지' 버튼</li></ul></div>
<img src="images/gcp_39_vm_stop_confirm.png" width="1050" alt="중지 확인 창. 빨간 상자: '중지' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 3 · 중지된 serving-vm 줄과 점 세 개 메뉴</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 다음 실습 날 점 세 개를 누르고 '시작/재개'를 고릅니다. 다시 켜면 외부 IP가 바뀔 수 있으니 목록에서 새 IP를 확인합니다(아래 고칠 곳).</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 상태 아이콘(중지됨), '시작/재개' 메뉴</li></ul></div>
<img src="images/gcp_40_vm_stopped_start.png" width="1050" alt="중지된 serving-vm 행(회색 정지 아이콘, 외부 IP 칸 비어 있음, SSH 흐림)과 점 세 개 메뉴. 빨간 상자: 상태 아이콘(중지됨), '시작/재개' 메뉴">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">다시 시작한 뒤 외부 IP가 바뀌었으면</strong> 아래 곳을 새 주소로 고칩니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">내 PC <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/.ssh/config</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">HostName</code>(6-3)</li>
<li style="margin:3px 0;">실습자료 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_HOST</code>(9절)</li>
<li style="margin:3px 0;">교안 03이나 과제 02를 시작했다면 두 GitHub 저장소(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-deploy</code>)의 변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_HOST</code>(Settings &gt; Secrets and variables &gt; Actions &gt; Variables)</li>
</ol>
</div>

### 10-2. 과정이 끝나면 삭제

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">과정이 끝나면 VM을 삭제합니다. 부팅 디스크도 함께 지워지고 임시 외부 IP도 반납됩니다. 방화벽 규칙 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">allow-serving-api</code>는 요금이 없어 남겨 두어도 됩니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 1 · VM 행 오른쪽 점 세 개(옵션 더보기) 메뉴</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 점 세 개를 누르고 '삭제'를 고릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> '삭제' 메뉴</li></ul></div>
<img src="images/gcp_35_vm_delete_menu.png" width="1050" alt="VM 행 오른쪽 점 세 개(옵션 더보기) 메뉴. 빨간 상자: '삭제' 메뉴">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 2 · 삭제 확인 창</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 부팅 디스크도 함께 지워진다는 문구를 읽고 '삭제'를 누릅니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> '부팅 디스크(serving-vm)가 삭제됩니다.' 문구, '삭제' 버튼</li></ul></div>
<img src="images/gcp_36_vm_delete_confirm.png" width="1050" alt="삭제 확인 창. 빨간 상자: '부팅 디스크(serving-vm)가 삭제됩니다.' 문구, '삭제' 버튼">

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">화면 3 · 삭제 후 빈 VM 인스턴스 목록</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;"><ul style="margin:4px 0 8px;padding-left:25px;"><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">할 일:</strong> 목록이 비었는지 확인합니다.</li><li style="margin:3px 0;"><strong style="font-size:1em;color:#1f3a5f;">빨간 상자:</strong> 빈 인스턴스 표, '인스턴스 삭제됨' 알림</li></ul></div>
<img src="images/gcp_37_vm_list_empty.png" width="1050" alt="삭제 후 빈 VM 인스턴스 목록과 '인스턴스 삭제됨' 알림. 빨간 상자: 빈 인스턴스 표, '인스턴스 삭제됨' 알림">

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">삭제한 뒤 왼쪽 메뉴 <strong style="font-size:1.13em;color:#1f3a5f;">스토리지 &gt; 스냅샷</strong>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vm</code> 디스크의 스냅샷이 남아 있으면 선택해 삭제합니다(VM을 만들 때 기본으로 켜지는 스냅샷 일정이 만든 사본으로, 저장 요금이 나올 수 있습니다).</p>
</div>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">출처:</strong> <a style="color:#245fa0;text-decoration:underline;" href="https://cloud.google.com/free/docs/free-cloud-features">GCP 무료 프로그램과 무료 등급</a>, <a style="color:#245fa0;text-decoration:underline;" href="https://cloud.google.com/billing/docs/how-to/payment-methods">결제 수단</a>, <a style="color:#245fa0;text-decoration:underline;" href="https://cloud.google.com/vpc/network-pricing">네트워크 가격(외부 IP)</a>, <a style="color:#245fa0;text-decoration:underline;" href="https://cloud.google.com/billing/docs/how-to/budgets">예산과 예산 알림</a>, <a style="color:#245fa0;text-decoration:underline;" href="https://cloud.google.com/compute/docs/connect/add-ssh-keys">SSH 키 추가</a>, <a style="color:#245fa0;text-decoration:underline;" href="https://cloud.google.com/firewall/docs/using-firewalls">방화벽 규칙</a>, <a style="color:#245fa0;text-decoration:underline;" href="https://docs.docker.com/engine/install/debian/">Docker Engine Debian 설치</a>, <a style="color:#245fa0;text-decoration:underline;" href="https://docs.docker.com/engine/install/linux-postinstall/">Linux 설치 후 설정</a></p>
</div>